# Claude Agent SDK · Python 从 0 到 1

> 来源：[Agent SDK overview](https://code.claude.com/docs/en/agent-sdk/overview)

> 一句话本质：**Client SDK 让你"调用模型"，Agent SDK 让你"雇一个会自己用工具干活的 Claude"**。前者你得自己写 tool loop，后者循环内建。

本笔记覆盖官方 Agent SDK 文档的全部主题（只讲 Python 版；TypeScript 仅在行为有差异时一句带过）。主线分四层递进：

1. **跑起来**（§3–§7）：安装认证 → 两个入口 → agent loop 原理 → 读消息流。
2. **控制它**（§8–§9、§11）：配置总线 → 权限系统与 HITL → Hooks。
3. **扩展它**（§10、§12–§14）：自定义工具 → 外部 MCP → 会话 → 子 agent。
4. **上生产**（§15–§23）：多轮与打断 → 流式输入/输出 → 结构化输出 → 系统提示 → Claude Code 文件系统特性 → 生产化控制与安全部署。


## 全局记忆图

先把整张地图记住，后面每一节都是往这张图上填细节。主线是：**为什么 → 两个入口 → 一份配置 → 一条消息流 → 两个控制面 → 四类扩展 → 生产化**。

```mermaid
flowchart TD
    A["为什么用 Agent SDK<br/>(不想手写 tool loop)"] --> B{选入口}
    B -->|一次性任务| Q["query()<br/>发任务→收消息流"]
    B -->|多轮/可打断| C["ClaudeSDKClient<br/>持续会话"]
    Q --> O["ClaudeAgentOptions<br/>(唯一的配置总线)"]
    C --> O
    O --> S["消息流 async for<br/>System/Assistant/User/Result"]
    S --> BLK["内容块<br/>Text / Thinking / ToolUse / ToolResult"]
    S -.可选.-> SE["StreamEvent 逐 token 流式输出"]
    S -.可选.-> SO["structured_output 结构化输出"]

    O ==控制面==> P["权限系统<br/>hooks→deny→ask→mode→allow→回调"]
    O ==控制面==> H["Hooks<br/>生命周期拦截/审计/改写"]

    O -.扩展.-> T2["自定义工具<br/>@tool + 进程内 MCP"]
    O -.扩展.-> T3["外部 MCP<br/>stdio / http / sse"]
    O -.扩展.-> T4["会话<br/>continue / resume / fork"]
    O -.扩展.-> T5["子 agent<br/>AgentDefinition"]
    O -.扩展.-> T6["文件系统特性<br/>CLAUDE.md / skills / plugins"]

    P & H --> PROD["生产化<br/>预算 / sandbox / 可观测 / 安全部署"]

    style A fill:#fef3c7,stroke:#d97706
    style O fill:#dbeafe,stroke:#2563eb
    style S fill:#dcfce7,stroke:#16a34a
    style BLK fill:#dcfce7,stroke:#16a34a
    style P fill:#fee2e2,stroke:#dc2626
    style H fill:#fee2e2,stroke:#dc2626
    style PROD fill:#ede9fe,stroke:#7c3aed
```

四个颜色对应四个"必须先懂"的核心：黄=动机，蓝=配置总线（所有能力都从 `ClaudeAgentOptions` 挂进去），绿=消息流（agent 干活的过程全靠读这条流理解），红=控制面（权限与 hooks 决定 agent 能做什么）。
